# k-fold cross-validation

One train/test split can be lucky or unlucky. **Cross-validation** scores a model on every part of the data, one part at a time, and gives you a mean and a spread.

This notebook goes with the **k-fold cross-validation** lesson on [ML Visual](https://mohammad-qt.github.io/ML_DL_Vis/). Run each cell in order with **Shift + Enter**. Nothing to install: Colab already has every library used here.

## 1. Load the data

Each row is a tumour sample from a breast biopsy. The columns are eight measurements of the cell nuclei (size, texture, shape). The goal: predict whether the sample is **malignant** (`diagnosis = 1`) or **benign** (`0`).

The data is the Breast Cancer Wisconsin (Diagnostic) data set from the UCI repository (CC BY 4.0), reduced to eight columns. This is a teaching example, not medical advice.

In [ ]:
import numpy as np
import pandas as pd

# The data lives in this project's GitHub repository, so there is nothing to upload.
DATA_URL = "https://raw.githubusercontent.com/Mohammad-QT/ML_DL_Vis/main/datasets/breast_cancer.csv"
dataset = pd.read_csv(DATA_URL)
X = dataset.drop(columns="diagnosis")
y = dataset["diagnosis"]
print(dataset.shape)
dataset.head()

## 2. One split is one opinion

We train the same model on five different random splits. Watch how much the test score moves.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

# The scaler lives inside the pipeline, so each fold fits it on its own training part only.
model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))

for seed in range(5):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=seed)
    print(f"split {seed}: test accuracy {model.fit(X_train, y_train).score(X_test, y_test):.3f}")

## 3. k-fold cross-validation

`KFold` cuts the rows into `n_splits` folds. `cross_val_score` trains one model per fold and returns the score of each held-out fold. `shuffle=True` mixes the rows first, and `random_state` makes the shuffle repeatable.

In [ ]:
from sklearn.model_selection import KFold, cross_val_score

cv = KFold(n_splits=5, shuffle=True, random_state=0)
scores = cross_val_score(model, X, y, cv=cv)
print("fold scores:", scores.round(3))
print(f"mean {scores.mean():.3f} +/- {scores.std():.3f}")

## 4. Compare models fairly

With the same folds, a cross-validated score is a fair way to compare two models.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

candidates = {
    "K-NN (k=5)": model,
    "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Decision tree (depth 4)": DecisionTreeClassifier(max_depth=4, random_state=0),
}
for name, m in candidates.items():
    s = cross_val_score(m, X, y, cv=cv)
    print(f"{name:26s} {s.mean():.3f} +/- {s.std():.3f}")

## 5. Try it yourself

- Change `n_splits` to 3 and to 10. What happens to the spread?
- Try `StratifiedKFold` from `sklearn.model_selection`. It keeps the share of each class the same in every fold.
- Remember: cross-validation only **measures**. Train your final model once on all the data.